# Literature Curve Kinetics Models

**Purpose.** This notebook is a **standalone, non-PCINN** workflow for fitting one small neural network per literature kinetics spreadsheet. It does **not** modify `/Users/justindong/Northeastern/Research/graduate/PCINN/MMA_PCINN.ipynb`, `/Users/justindong/Northeastern/Research/graduate/PCINN/MMA_PCINN.py`, or anything under `/Users/justindong/Northeastern/Research/graduate/PCINN/exports/`.

**Goal.** For each spreadsheet, learn the literature curve mapping:
- input 1: series label (`45`, `55`, `60`, etc.)
- input 2: x-axis value (`t_res`, `time(min)`, or `time(h)`)
- output: `-ln(1-conversion)`

This mirrors the notebook-driven style of the main PCINN notebook, but keeps the new work isolated so the original project remains untouched.


In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from literature_curve_kinetics_models import (
    ARTIFACTS_DIR,
    DATA_DIR,
    axis_label_to_column_name,
    parse_curve_workbook,
    preview_predictions,
    run_all_models,
)


## Reproducibility & Paths

- Extracted `.xlsx` files live in `literature_curve_xlsx/`
- All saved outputs live in `literature_curve_artifacts/`
- This workflow stays isolated from the original PCINN files so we can iterate safely

**Training hyperparameters**
- epochs: `4000`
- learning rate: `0.01`
- weight decay: `1e-4`
- random seed: `42`


In [ ]:
print("Data directory:", DATA_DIR)
print("Artifacts directory:", ARTIFACTS_DIR)
print("\nWorkbook files:")
for path in sorted(DATA_DIR.glob("*.xlsx")):
    print(" -", path.name)


## Data Preprocessing

Each workbook is stored as paired columns in the digitized spreadsheet:
- the column header is the temperature (`45`, `55`, `60`, etc.)
- row 1 of that pair stores the x/y axis labels
- rows 2+ store the numeric digitized points

The helper `parse_curve_workbook(...)` reshapes each workbook into a tidy training table with:
- `source_file`
- `system`
- `series_label` (used internally, then displayed as `Temp (degree Celsius)`)
- `x_label`
- `y_label`
- `x_value`
- `y_value`

Before training, rows where both the x-axis value and `-ln(1-conversion)` are zero are removed. Those origin points do not add useful signal for the fitted curves.

When predictions are saved, the output columns are renamed for readability:
- temperature column becomes `Temp (degree Celsius)`
- literature target becomes `-ln(1-conversion) (expected)`
- model output becomes `-ln(1-conversion) (actual)`


In [ ]:
example = parse_curve_workbook(DATA_DIR / 'MMA_fig1a.xlsx').copy()
example['Temp (degree Celsius)'] = example['series_label'].map(lambda value: f'{value:.2f}')
example_display = example.rename(columns={'x_value': 't_res', 'y_value': '-ln(1-conversion)'})
example_display = example_display[['Temp (degree Celsius)', 't_res', '-ln(1-conversion)']]
example_display.head(12)


## Model Definitions

The standalone trainer uses a small two-input multilayer perceptron:
- inputs: `[series_label, workbook x-axis value]`
- network: `Linear(2, 32) -> tanh -> Linear(32, 32) -> tanh -> Linear(32, 1)`
- output: predicted `-ln(1-conversion)`

Before training, both inputs and targets are standardized. In the saved prediction tables, the digitized literature values are shown as `-ln(1-conversion) (expected)` and the neural-network outputs are shown as `-ln(1-conversion) (actual)`.


## Train All Workbook-Specific Models

Run the next cell to retrain all four isolated models. It will regenerate the saved metrics, point predictions, dense predictions, plots, and model weights.


In [ ]:
summary = run_all_models()
summary


## Latest Saved Metrics

These metrics come from the current saved artifact set in `literature_curve_artifacts/summary_metrics.csv`. They measure how closely each workbook-specific model fits its own digitized points after removing `(0, 0)` rows.

| source_file | system | x_label | n_points | rmse | mae | r2 |
| --- | --- | --- | --- | --- | --- | --- |
| MMA_fig1a.xlsx | MMA | t_res | 242 | 0.002135 | 0.001694 | 0.994513 |
| MMA_fig2a.xlsx | MMA | time(min) | 16 | 0.018799 | 0.015174 | 0.997047 |
| STY_fig1b.xlsx | STY | t_res | 242 | 0.001859 | 0.001481 | 0.969967 |
| STY_fig2c.xlsx | STY | time(h) | 15 | 0.000394 | 0.000330 | 0.999994 |


## Workbook Fit Plots

These are the saved fit diagnostics. Each left panel overlays the digitized literature points with the neural-network prediction for that workbook, and the right panel shows training loss.

### MMA_fig1a
![MMA_fig1a fit](literature_curve_artifacts/plots/MMA_fig1a_fit.png)

### MMA_fig2a
![MMA_fig2a fit](literature_curve_artifacts/plots/MMA_fig2a_fit.png)

### STY_fig1b
![STY_fig1b fit](literature_curve_artifacts/plots/STY_fig1b_fit.png)

### STY_fig2c
![STY_fig2c fit](literature_curve_artifacts/plots/STY_fig2c_fit.png)


## Sample Predicted Values

Each table below shows the first **three rows from each temperature**. The naming is now shorter and easier to scan:

- `-ln(1-conversion) (expected)` = digitized literature value
- `-ln(1-conversion) (actual)` = neural-network output
- `actual_minus_expected` = model output minus literature value

### MMA_fig1a predictions

| Temp (degree Celsius) | t_res | -ln(1-conversion) (expected) | -ln(1-conversion) (actual) | actual_minus_expected |
| --- | --- | --- | --- | --- |
| 60.00 | 1.385681 | 0.006303 | 0.007636 | 0.001334 |
| 60.00 | 1.778291 | 0.008403 | 0.009891 | 0.001487 |
| 60.00 | 2.217090 | 0.012185 | 0.012587 | 0.000402 |
| 70.00 | 0.946882 | 0.010504 | 0.010822 | 0.000317 |
| 70.00 | 1.824480 | 0.013025 | 0.016150 | 0.003125 |
| 70.00 | 2.055427 | 0.015966 | 0.017730 | 0.001763 |
| 80.00 | 0.993072 | 0.014286 | 0.013667 | -0.000619 |
| 80.00 | 1.039261 | 0.015126 | 0.013924 | -0.001202 |
| 80.00 | 1.200924 | 0.013866 | 0.014857 | 0.000991 |
| 90.00 | 0.969977 | 0.007563 | 0.007471 | -0.000092 |
| 90.00 | 1.131640 | 0.008824 | 0.009185 | 0.000361 |
| 90.00 | 1.247113 | 0.010084 | 0.010474 | 0.000390 |

### MMA_fig2a predictions

| Temp (degree Celsius) | time_min | -ln(1-conversion) (expected) | -ln(1-conversion) (actual) | actual_minus_expected |
| --- | --- | --- | --- | --- |
| 45.00 | 9.146341 | 0.109787 | 0.123978 | 0.014191 |
| 45.00 | 19.512195 | 0.171064 | 0.184025 | 0.012961 |
| 45.00 | 59.756098 | 0.362553 | 0.366855 | 0.004302 |
| 55.00 | 9.146341 | 0.102128 | 0.106235 | 0.004107 |
| 55.00 | 19.512195 | 0.171064 | 0.176267 | 0.005203 |
| 55.00 | 39.634146 | 0.275745 | 0.275200 | -0.000545 |
| 65.00 | 0.000000 | 0.002553 | -0.005361 | -0.007914 |
| 65.00 | 9.756098 | 0.109787 | 0.101590 | -0.008197 |
| 65.00 | 29.878049 | 0.326809 | 0.308931 | -0.017877 |

### STY_fig1b predictions

| Temp (degree Celsius) | t_res | -ln(1-conversion) (expected) | -ln(1-conversion) (actual) | actual_minus_expected |
| --- | --- | --- | --- | --- |
| 50.00 | 0.990676 | 0.015117 | 0.012711 | -0.002405 |
| 50.00 | 1.048951 | 0.011890 | 0.012864 | 0.000974 |
| 50.00 | 1.130536 | 0.011380 | 0.013082 | 0.001702 |
| 60.00 | 1.002331 | 0.019873 | 0.015848 | -0.004025 |
| 60.00 | 1.060606 | 0.013758 | 0.016069 | 0.002311 |
| 60.00 | 1.130536 | 0.012909 | 0.016341 | 0.003432 |
| 70.00 | 0.990676 | 0.020552 | 0.019553 | -0.000999 |
| 70.00 | 1.072261 | 0.020552 | 0.019909 | -0.000643 |
| 70.00 | 1.118881 | 0.019703 | 0.020118 | 0.000415 |
| 80.00 | 0.990676 | 0.027176 | 0.024886 | -0.002290 |
| 80.00 | 1.060606 | 0.024289 | 0.025243 | 0.000955 |
| 80.00 | 1.130536 | 0.024628 | 0.025608 | 0.000980 |

### STY_fig2c predictions

| Temp (degree Celsius) | time_h | -ln(1-conversion) (expected) | -ln(1-conversion) (actual) | actual_minus_expected |
| --- | --- | --- | --- | --- |
| 55.00 | 3.988764 | 0.056842 | 0.056961 | 0.000119 |
| 55.00 | 9.466292 | 0.186316 | 0.186363 | 0.000047 |
| 55.00 | 16.966292 | 0.296842 | 0.296517 | -0.000326 |
| 65.00 | 3.960674 | 0.173684 | 0.173152 | -0.000532 |
| 65.00 | 9.971910 | 0.286737 | 0.287241 | 0.000504 |
| 65.00 | 15.955056 | 0.444632 | 0.444273 | -0.000359 |
| 75.00 | 0.000000 | 0.017684 | 0.018124 | 0.000440 |
| 75.00 | 2.977528 | 0.179368 | 0.179105 | -0.000264 |
| 75.00 | 5.955056 | 0.289263 | 0.289069 | -0.000195 |



In [ ]:
previews = preview_predictions()
for name, frame in previews.items():
    print(f"\n{name} (first 3 rows per temperature)")
    display(frame)


## Saved Outputs

After training, inspect these paths:
- `literature_curve_artifacts/summary_metrics.csv`
- `literature_curve_artifacts/predictions/*_point_predictions.csv`
- `literature_curve_artifacts/predictions/*_dense_predictions.csv`
- `literature_curve_artifacts/plots/*_fit.png`
- `literature_curve_artifacts/models/*_model.pt`

The prediction files now use shorter labels such as `Temp (degree Celsius)`, `-ln(1-conversion) (expected)`, and `-ln(1-conversion) (actual)` so the tables read more like the original figures.


## Extension Notes for Future Researchers

This notebook is the simplest safe first step: fit one kinetics-only model per workbook.

Good next steps:
1. leave-one-series-out or leave-one-temperature-out validation
2. a second isolated workflow for `conversion -> Mn / PDI`
3. interpolation logic to fuse kinetics and molecular-weight tables for a richer supervised dataset

The key rule is to keep this workflow isolated from the original PCINN notebook unless you intentionally decide to integrate them later.
